# core.windowing

> Pure math functions for viewport window and scrollbar calculations.

In [ ]:
#| default_exp core.windowing

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Tuple

## Window Calculation

In [ ]:
#| export
def clamp_window_start(window_start: int,  # Requested first visible row
                       visible_rows: int,   # Number of visible rows
                       total_items: int,     # Total item count
                      ) -> int:              # Clamped window_start
    """Clamp window_start to valid range."""
    if total_items <= 0: return 0
    max_start = max(0, total_items - visible_rows)
    return max(0, min(window_start, max_start))

In [ ]:
#| export
def compute_window(window_start: int,   # First visible row index (already clamped)
                   visible_rows: int,    # Number of visible rows
                   total_items: int,     # Total item count
                  ) -> Tuple[int, int]:  # (render_start, render_end) exclusive end
    """Compute the range of rows to render."""
    if total_items <= 0: return (0, 0)
    render_start = max(0, window_start)
    render_end = min(total_items, window_start + visible_rows)
    return (render_start, render_end)

## Scrollbar Calculation

In [ ]:
#| export
def compute_scrollbar(window_start: int,      # First visible row index
                      visible_rows: int,       # Number of visible rows
                      total_items: int,        # Total item count
                      track_height: float,     # Scrollbar track height in px
                      min_thumb_height: int = 24,  # Minimum thumb height in px
                     ) -> Tuple[float, float]: # (thumb_top_percent, thumb_height_percent)
    """Compute scrollbar thumb position and size as percentages."""
    if total_items <= 0 or visible_rows >= total_items:
        return (0.0, 100.0)  # Thumb fills entire track

    # Thumb height as proportion of visible/total
    thumb_height_pct = (visible_rows / total_items) * 100.0

    # Enforce minimum thumb height
    if track_height > 0:
        min_pct = (min_thumb_height / track_height) * 100.0
        thumb_height_pct = max(thumb_height_pct, min_pct)

    # Thumb position
    max_start = max(1, total_items - visible_rows)
    thumb_top_pct = (window_start / max_start) * (100.0 - thumb_height_pct)

    return (thumb_top_pct, thumb_height_pct)

## Navigation Helpers

In [ ]:
#| export
def navigate(window_start: int,   # Current first visible row
             direction: str,       # 'up', 'down', 'page_up', 'page_down', 'first', 'last'
             visible_rows: int,    # Number of visible rows
             total_items: int,     # Total item count
            ) -> int:              # New window_start (clamped)
    """Compute new window_start for a navigation action."""
    if direction == 'up':
        new_start = window_start - 1
    elif direction == 'down':
        new_start = window_start + 1
    elif direction == 'page_up':
        page_size = max(1, visible_rows - 1)
        new_start = window_start - page_size
    elif direction == 'page_down':
        page_size = max(1, visible_rows - 1)
        new_start = window_start + page_size
    elif direction == 'first':
        new_start = 0
    elif direction == 'last':
        new_start = max(0, total_items - visible_rows)
    else:
        raise ValueError(f"Unknown direction: {direction}")

    return clamp_window_start(new_start, visible_rows, total_items)

In [ ]:
#| export
def compute_visible_rows(viewport_height: float,  # Available height in px
                         row_height: int,          # Row height in px
                        ) -> int:                  # Number of visible rows
    """Compute how many rows fit in the viewport."""
    if row_height <= 0 or viewport_height <= 0: return 0
    return max(1, int(viewport_height // row_height))

## Tests

In [ ]:
# Test clamp_window_start
assert clamp_window_start(0, 15, 500) == 0
assert clamp_window_start(490, 15, 500) == 485  # 500-15=485
assert clamp_window_start(-5, 15, 500) == 0
assert clamp_window_start(0, 15, 0) == 0  # empty collection
assert clamp_window_start(0, 15, 10) == 0  # fewer items than visible
print("clamp_window_start tests passed")

clamp_window_start tests passed


In [ ]:
# Test compute_window
assert compute_window(0, 15, 500) == (0, 15)
assert compute_window(485, 15, 500) == (485, 500)
assert compute_window(0, 15, 0) == (0, 0)  # empty
assert compute_window(0, 15, 10) == (0, 10)  # fewer items than visible
print("compute_window tests passed")

compute_window tests passed


In [ ]:
# Test compute_scrollbar
# All items visible -> full track
top, height = compute_scrollbar(0, 15, 10, 600)
assert top == 0.0 and height == 100.0

# At start (min_thumb_height=0 to test pure proportional math)
top, height = compute_scrollbar(0, 15, 500, 600, min_thumb_height=0)
assert top == 0.0
assert height == (15 / 500) * 100  # 3%

# At start with default min_thumb_height (24px on 600px track = 4%)
top, height = compute_scrollbar(0, 15, 500, 600)
assert top == 0.0
assert height == (24 / 600) * 100  # min_thumb enforced: 4%

# At end
top, height = compute_scrollbar(485, 15, 500, 600)
assert abs(top + height - 100.0) < 0.01  # thumb reaches bottom

# Minimum thumb height enforcement
top, height = compute_scrollbar(0, 1, 10000, 600, min_thumb_height=24)
assert height >= (24 / 600) * 100  # at least min height

print("compute_scrollbar tests passed")

compute_scrollbar tests passed


In [ ]:
# Test navigate
assert navigate(10, 'up', 15, 500) == 9
assert navigate(10, 'down', 15, 500) == 11
assert navigate(0, 'up', 15, 500) == 0  # clamped
assert navigate(485, 'down', 15, 500) == 485  # clamped
assert navigate(10, 'page_up', 15, 500) == 0  # 10 - 14 = -4 → 0
assert navigate(10, 'page_down', 15, 500) == 24  # 10 + 14 = 24
assert navigate(10, 'first', 15, 500) == 0
assert navigate(10, 'last', 15, 500) == 485
print("navigate tests passed")

navigate tests passed


In [ ]:
# Test compute_visible_rows
assert compute_visible_rows(600, 40) == 15  # 600/40 = 15
assert compute_visible_rows(610, 40) == 15  # floor(610/40) = 15
assert compute_visible_rows(30, 40) == 1    # minimum 1
assert compute_visible_rows(0, 40) == 0     # no viewport
assert compute_visible_rows(600, 0) == 0    # zero row height
print("compute_visible_rows tests passed")

compute_visible_rows tests passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()